# Chapter 17: Memory Systems — Patient History for the GAS Advisor

> **Part 5: Agentic AI Systems** | Guardian Angel System (GAS) — Multi-Agent Diabetic Patient Supervision

---

## 🎯 Chapter Goal

By the end of this chapter you will:

1. **Implement** all 4 memory types for the GAS system: Working, Episodic, Semantic, Procedural
2. **Build** memory consolidation: summarize a day's readings into one episodic memory
3. **Implement** temporal decay: recent memories score higher than old ones
4. **Simulate** 3 days of patient data and watch memory accumulate

### Why Memory Matters for Medical AI

Without memory, every interaction starts from scratch. The GAS Advisor Agent would not know:
- That Alice had a severe hypoglycemia episode last Tuesday at 3 AM
- That her glucose always spikes after pizza (high-fat meal effect)
- That she prefers glucose tablets over juice for treating lows
- That her insulin sensitivity drops by 30% during her menstrual cycle

Memory transforms a stateless Q&A system into a **personalized clinical companion**.

## 📖 Theory Recap

### The 4 Memory Types

| Type | Storage | Retrieval | GAS Use Case |
|------|---------|-----------|-------------|
| **Working** | In-context (LLM window) | Immediate | Last 2h of CGM readings |
| **Episodic** | Vector store (Qdrant/FAISS) | Similarity search | Past hypo/hyper episodes |
| **Semantic** | Knowledge graph / dict | Key lookup | Patient profile, preferences |
| **Procedural** | Model weights (fine-tuning) | Implicit | Clinical reasoning patterns |

### Memory Consolidation

Working memory is limited by the context window. **Consolidation** moves important information to long-term storage:

```
Working Memory (last 2h readings)
        │
        │ [every 24h]
        ▼
Consolidation: summarize day → one episodic memory
        │
        ▼
Episodic Memory (vector store)
```

### Temporal Decay

Not all memories are equally relevant. A hypoglycemia episode from yesterday is more relevant than one from 6 months ago.

**Temporal decay formula**:
$$\text{score}(d) = \text{similarity}(q, d) \times e^{-\lambda \cdot \Delta t}$$

where:
- $\text{similarity}(q, d)$ = cosine similarity between query and memory
- $\lambda$ = decay rate (e.g., 0.1 per day)
- $\Delta t$ = days since the memory was created

### Memory for Multi-Agent Systems

In GAS, memory is **shared vs private**:

| Memory | Shared? | Who Reads | Who Writes |
|--------|---------|-----------|------------|
| Working (GASState) | ✅ Shared | All agents | Monitor, Advisor |
| Episodic (vector store) | ✅ Shared | Advisor | Monitor, Advisor |
| Semantic (patient profile) | ✅ Shared | All agents | Supervisor |
| Procedural (model weights) | ❌ Private | Implicit | Training pipeline |

## 🔨 Build It

### Step 1: Install Dependencies

In [ ]:
# Install required packages
# %pip install numpy python-dotenv

### Step 2: Environment Setup

In [ ]:
import os
import json
import math
import random
import datetime
import numpy as np
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field
from collections import deque

# TEST_MODE: set NOTEBOOK_TEST_MODE=1 to run without real API keys or heavy models
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "../.."))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL

print(f"TEST_MODE = {TEST_MODE}, BACKEND = {BACKEND}")

# Reproducible random seed
random.seed(42)
np.random.seed(42)

### Step 3: Reuse GASState Schema from Chapter 15

In [ ]:
# ─── Core Data Classes (from Chapter 15) ─────────────────────────────────────

@dataclass
class PatientProfile:
    patient_id: str
    name: str
    age: int
    diabetes_type: str
    target_glucose_low: float
    target_glucose_high: float
    insulin_sensitivity: float
    carb_ratio: float
    emergency_contact: str = "caregiver@example.com"


@dataclass
class GlucoseReading:
    value: float
    timestamp: datetime.datetime
    trend: str
    sensor_id: str
    confidence: float = 1.0


# Alice — our synthetic T1D patient
alice = PatientProfile(
    patient_id="patient-001",
    name="Alice",
    age=34,
    diabetes_type="T1D",
    target_glucose_low=70.0,
    target_glucose_high=180.0,
    insulin_sensitivity=50.0,
    carb_ratio=15.0,
    emergency_contact="bob.smith@example.com"
)

print(f"✅ Patient: {alice.name}, {alice.age}yo {alice.diabetes_type}")

### Step 4: Memory Type 1 — Working Memory (Sliding Window)

In [ ]:
# ─── Working Memory: Sliding Window ──────────────────────────────────────────

class WorkingMemory:
    """
    Working memory = last N glucose readings in the LLM context window.
    Implemented as a sliding window (deque) with a fixed capacity.
    """

    def __init__(self, capacity: int = 24):  # 24 readings = 2 hours at 5-min intervals
        self.capacity = capacity
        self._readings: deque = deque(maxlen=capacity)

    def add(self, reading: GlucoseReading):
        """Add a new reading, evicting the oldest if at capacity."""
        self._readings.append(reading)

    def get_recent(self, n: int = None) -> List[GlucoseReading]:
        """Get the n most recent readings (all if n is None)."""
        readings = list(self._readings)
        if n:
            return readings[-n:]
        return readings

    def to_context_string(self) -> str:
        """Format readings as a string for LLM context."""
        if not self._readings:
            return "No recent readings available."
        lines = ["Recent glucose readings (newest last):"]
        for r in self._readings:
            time_str = r.timestamp.strftime("%H:%M")
            lines.append(f"  {time_str}: {r.value:.0f} mg/dL ({r.trend})")
        return "\n".join(lines)

    def get_stats(self) -> dict:
        """Compute statistics over the working memory window."""
        if not self._readings:
            return {}
        values = [r.value for r in self._readings]
        return {
            "count": len(values),
            "mean": round(np.mean(values), 1),
            "min": round(min(values), 1),
            "max": round(max(values), 1),
            "std": round(np.std(values), 1),
            "trend": "rising" if values[-1] > values[0] else "falling" if values[-1] < values[0] else "stable"
        }

    def __len__(self):
        return len(self._readings)


# Demo: Working memory with 2 hours of readings
working_mem = WorkingMemory(capacity=24)

# Simulate 2 hours of readings (every 5 minutes)
base_time = datetime.datetime.now() - datetime.timedelta(hours=2)
glucose_values = [140, 145, 150, 148, 142, 138, 130, 122, 115, 108,
                  100, 95, 88, 82, 78, 74, 70, 68, 65, 62, 58, 55, 50, 45]

for i, val in enumerate(glucose_values):
    ts = base_time + datetime.timedelta(minutes=i*5)
    trend = "falling" if i > 5 else "stable"
    working_mem.add(GlucoseReading(val, ts, trend, "dexcom-g7-001"))

print("📊 Working Memory (last 2 hours):")
print(working_mem.to_context_string())
print(f"\nStats: {working_mem.get_stats()}")

### Step 5: Memory Type 2 — Episodic Memory (Vector Store)

In [ ]:
# ─── Episodic Memory: Vector-Indexed Episodes ─────────────────────────────────

class EpisodicMemory:
    """
    Episodic memory stores past clinical episodes (hypoglycemia, hyperglycemia, etc.)
    indexed by embedding for similarity search.

    In production, this would use Qdrant or FAISS.
    Here we use an in-memory implementation with numpy.
    """

    def __init__(self, embedding_dim: int = 64):
        self.embedding_dim = embedding_dim
        self.episodes: List[dict] = []
        self.embeddings: List[np.ndarray] = []

    def _embed_episode(self, episode: dict) -> np.ndarray:
        """
        Create a deterministic mock embedding for an episode.
        In production: use sentence-transformers on episode description.
        """
        text = f"{episode.get('type', '')} {episode.get('description', '')} {episode.get('glucose_value', '')}"
        seed = sum(ord(c) for c in text) % 100000
        np.random.seed(seed)
        vec = np.random.randn(self.embedding_dim).astype(np.float32)
        return vec / np.linalg.norm(vec)

    def store_episode(self, episode: dict):
        """
        Store a clinical episode with its embedding.

        episode dict should contain:
          - type: "hypoglycemia" | "hyperglycemia" | "normal" | "dka_risk"
          - description: text description
          - glucose_value: float
          - timestamp: datetime
          - outcome: what happened / how it was resolved
          - severity: "mild" | "moderate" | "severe"
        """
        embedding = self._embed_episode(episode)
        self.episodes.append(episode)
        self.embeddings.append(embedding)

    def recall_similar_episodes(
        self,
        current_state: dict,
        k: int = 3,
        decay_lambda: float = 0.1
    ) -> List[dict]:
        """
        Retrieve k most similar past episodes with temporal decay.

        Score = cosine_similarity * exp(-lambda * days_ago)
        """
        if not self.episodes:
            return []

        # Embed current state
        query_embedding = self._embed_episode(current_state)

        # Compute scores with temporal decay
        now = datetime.datetime.now()
        scored = []
        for i, (episode, emb) in enumerate(zip(self.episodes, self.embeddings)):
            # Cosine similarity
            similarity = float(np.dot(query_embedding, emb))

            # Temporal decay
            episode_time = episode.get("timestamp", now)
            if isinstance(episode_time, str):
                episode_time = datetime.datetime.fromisoformat(episode_time)
            days_ago = (now - episode_time).total_seconds() / 86400
            decay = math.exp(-decay_lambda * days_ago)

            final_score = similarity * decay
            scored.append((i, final_score, similarity, decay, days_ago))

        # Sort by final score
        scored.sort(key=lambda x: x[1], reverse=True)

        # Return top-k with scores
        results = []
        for idx, score, sim, decay, days_ago in scored[:k]:
            ep = dict(self.episodes[idx])
            ep["_retrieval_score"] = round(score, 4)
            ep["_similarity"] = round(sim, 4)
            ep["_decay"] = round(decay, 4)
            ep["_days_ago"] = round(days_ago, 1)
            results.append(ep)

        return results

    def __len__(self):
        return len(self.episodes)


print("✅ EpisodicMemory class defined")

In [ ]:
# ─── Populate Episodic Memory with Past Episodes ──────────────────────────────

episodic_mem = EpisodicMemory(embedding_dim=64)

# Simulate 3 days of past episodes
now = datetime.datetime.now()

past_episodes = [
    {
        "type": "hypoglycemia",
        "description": "Nocturnal hypoglycemia after evening exercise. Glucose dropped to 48 mg/dL at 3 AM.",
        "glucose_value": 48.0,
        "timestamp": now - datetime.timedelta(days=3, hours=3),
        "outcome": "Treated with 15g glucose tablets. Recovered to 95 mg/dL in 20 minutes.",
        "severity": "severe",
        "trigger": "evening_exercise"
    },
    {
        "type": "hyperglycemia",
        "description": "Post-pizza hyperglycemia. Glucose rose to 285 mg/dL 3 hours after dinner.",
        "glucose_value": 285.0,
        "timestamp": now - datetime.timedelta(days=2, hours=5),
        "outcome": "Correction dose of 3 units given. Glucose normalized in 3 hours.",
        "severity": "moderate",
        "trigger": "high_fat_meal"
    },
    {
        "type": "hypoglycemia",
        "description": "Pre-lunch hypoglycemia. Glucose at 62 mg/dL before lunch.",
        "glucose_value": 62.0,
        "timestamp": now - datetime.timedelta(days=2, hours=1),
        "outcome": "Consumed 4 oz orange juice. Glucose rose to 98 mg/dL in 15 minutes.",
        "severity": "mild",
        "trigger": "delayed_meal"
    },
    {
        "type": "hyperglycemia",
        "description": "Dawn phenomenon. Glucose rose from 110 to 165 mg/dL between 4-7 AM.",
        "glucose_value": 165.0,
        "timestamp": now - datetime.timedelta(days=1, hours=7),
        "outcome": "No intervention needed. Glucose within target by 8 AM after breakfast bolus.",
        "severity": "mild",
        "trigger": "dawn_phenomenon"
    },
    {
        "type": "hypoglycemia",
        "description": "Exercise-induced hypoglycemia during afternoon run. Glucose at 55 mg/dL.",
        "glucose_value": 55.0,
        "timestamp": now - datetime.timedelta(hours=18),
        "outcome": "Stopped exercise, consumed 20g carbs. Glucose recovered to 85 mg/dL.",
        "severity": "moderate",
        "trigger": "aerobic_exercise"
    },
    {
        "type": "hypoglycemia",
        "description": "Overnight hypoglycemia. Glucose fell to 45 mg/dL at 2 AM.",
        "glucose_value": 45.0,
        "timestamp": now - datetime.timedelta(hours=10),
        "outcome": "CGM alarm woke patient. Treated with glucose tablets.",
        "severity": "severe",
        "trigger": "nocturnal"
    }
]

for ep in past_episodes:
    episodic_mem.store_episode(ep)

print(f"✅ Stored {len(episodic_mem)} episodes in episodic memory")
print("\nEpisode summary:")
for ep in past_episodes:
    days_ago = (now - ep["timestamp"]).total_seconds() / 3600
    print(f"  [{ep['type']:15s}] {ep['severity']:8s} | {days_ago:.1f}h ago | {ep['description'][:60]}...")

In [ ]:
# ─── Test Episodic Memory Recall ──────────────────────────────────────────────

# Current state: glucose is 45 mg/dL (critical hypoglycemia)
current_state = {
    "type": "hypoglycemia",
    "description": "Critical hypoglycemia. Glucose at 45 mg/dL with falling trend.",
    "glucose_value": 45.0,
    "timestamp": now
}

print("🔍 Episodic Memory Recall — Current: glucose=45 mg/dL (critical hypoglycemia)")
print("=" * 65)

recalled = episodic_mem.recall_similar_episodes(current_state, k=3, decay_lambda=0.1)

for i, ep in enumerate(recalled, 1):
    print(f"\n{i}. {ep['type'].upper()} — {ep['severity']} severity")
    print(f"   Description: {ep['description'][:80]}")
    print(f"   Outcome: {ep['outcome'][:80]}")
    print(f"   Retrieval: score={ep['_retrieval_score']:.4f} "
          f"(sim={ep['_similarity']:.4f} × decay={ep['_decay']:.4f}, {ep['_days_ago']:.1f} days ago)")

### Step 6: Memory Type 3 — Semantic Memory (Knowledge Graph)

In [ ]:
# ─── Semantic Memory: Patient Knowledge Graph ─────────────────────────────────

class SemanticMemory:
    """
    Semantic memory stores structured facts about the patient.
    Implemented as a nested dict (knowledge graph).

    In production: use a graph database (Neo4j) or structured JSON.
    """

    def __init__(self, patient: PatientProfile):
        self._graph: Dict[str, Any] = {}
        self._initialize_from_profile(patient)

    def _initialize_from_profile(self, patient: PatientProfile):
        """Initialize semantic memory from patient profile."""
        self._graph = {
            "patient": {
                "id": patient.patient_id,
                "name": patient.name,
                "age": patient.age,
                "diabetes_type": patient.diabetes_type,
            },
            "clinical": {
                "target_range": [patient.target_glucose_low, patient.target_glucose_high],
                "insulin_sensitivity": patient.insulin_sensitivity,
                "carb_ratio": patient.carb_ratio,
                "a1c_last": None,
                "tir_last_30d": None,
            },
            "preferences": {
                "hypo_treatment": "glucose_tablets",  # preferred treatment
                "notification_method": "sms",
                "language": "en",
            },
            "patterns": {
                "dawn_phenomenon": False,
                "exercise_sensitivity": "high",
                "hypoglycemia_unawareness": True,  # Alice has HU
                "typical_hypo_time": "nocturnal",
            },
            "medications": {
                "insulin_type": "rapid_acting_aspart",
                "basal_insulin": "glargine",
                "delivery": "MDI",  # Multiple Daily Injections
                "other_meds": []
            },
            "contacts": {
                "emergency": patient.emergency_contact,
                "endocrinologist": "dr.chen@clinic.com",
                "cde": "nurse.jones@clinic.com"
            }
        }

    def get(self, path: str, default=None) -> Any:
        """Get a value by dot-separated path (e.g., 'clinical.insulin_sensitivity')."""
        keys = path.split(".")
        node = self._graph
        for key in keys:
            if isinstance(node, dict) and key in node:
                node = node[key]
            else:
                return default
        return node

    def set(self, path: str, value: Any):
        """Set a value by dot-separated path."""
        keys = path.split(".")
        node = self._graph
        for key in keys[:-1]:
            node = node.setdefault(key, {})
        node[keys[-1]] = value

    def update_from_episode(self, episode: dict):
        """Update semantic memory based on a new episode."""
        ep_type = episode.get("type", "")
        trigger = episode.get("trigger", "")

        # Update patterns
        if trigger == "dawn_phenomenon":
            self.set("patterns.dawn_phenomenon", True)
        if trigger in ["aerobic_exercise", "evening_exercise"]:
            self.set("patterns.exercise_sensitivity", "high")
        if trigger == "nocturnal" and ep_type == "hypoglycemia":
            self.set("patterns.typical_hypo_time", "nocturnal")

    def to_context_string(self) -> str:
        """Format semantic memory as a string for LLM context."""
        p = self._graph
        lines = [
            f"Patient: {p['patient']['name']}, {p['patient']['age']}yo {p['patient']['diabetes_type']}",
            f"Target range: {p['clinical']['target_range'][0]}-{p['clinical']['target_range'][1]} mg/dL",
            f"ISF: {p['clinical']['insulin_sensitivity']} mg/dL/unit, ICR: 1:{p['clinical']['carb_ratio']}",
            f"Hypoglycemia unawareness: {p['patterns']['hypoglycemia_unawareness']}",
            f"Dawn phenomenon: {p['patterns']['dawn_phenomenon']}",
            f"Exercise sensitivity: {p['patterns']['exercise_sensitivity']}",
            f"Preferred hypo treatment: {p['preferences']['hypo_treatment']}",
            f"Delivery: {p['medications']['delivery']}",
        ]
        return "\n".join(lines)


# Initialize semantic memory for Alice
semantic_mem = SemanticMemory(alice)

# Update from past episodes
for ep in past_episodes:
    semantic_mem.update_from_episode(ep)

print("🧠 Semantic Memory (Patient Knowledge Graph):")
print(semantic_mem.to_context_string())
print(f"\nHypoglycemia unawareness: {semantic_mem.get('patterns.hypoglycemia_unawareness')}")
print(f"Emergency contact: {semantic_mem.get('contacts.emergency')}")

### Step 7: Memory Consolidation — Daily Summary

In [ ]:
# ─── Memory Consolidation ─────────────────────────────────────────────────────

def consolidate_daily_readings(
    readings: List[GlucoseReading],
    date: datetime.date,
    patient: PatientProfile
) -> dict:
    """
    Consolidate a day's glucose readings into a single episodic memory.

    This is the memory consolidation step: working memory → episodic memory.
    After 24h, we summarize the day's readings into one structured episode.
    """
    if not readings:
        return {}

    values = [r.value for r in readings]
    timestamps = [r.timestamp for r in readings]

    # Compute time-in-range metrics
    tir = sum(1 for v in values if patient.target_glucose_low <= v <= patient.target_glucose_high)
    tbr = sum(1 for v in values if v < patient.target_glucose_low)
    tar = sum(1 for v in values if v > patient.target_glucose_high)
    total = len(values)

    # Detect episodes
    hypo_episodes = []
    hyper_episodes = []
    in_hypo = False
    in_hyper = False

    for r in readings:
        if r.value < 70 and not in_hypo:
            hypo_episodes.append({"start": r.timestamp, "min_value": r.value})
            in_hypo = True
        elif r.value >= 70:
            in_hypo = False

        if r.value > 180 and not in_hyper:
            hyper_episodes.append({"start": r.timestamp, "max_value": r.value})
            in_hyper = True
        elif r.value <= 180:
            in_hyper = False

    # Build summary
    summary = {
        "type": "daily_summary",
        "date": date.isoformat(),
        "timestamp": datetime.datetime.combine(date, datetime.time(23, 59)),
        "description": (
            f"Day summary for {date}. "
            f"Mean glucose: {np.mean(values):.0f} mg/dL. "
            f"TIR: {tir/total*100:.0f}%, TBR: {tbr/total*100:.0f}%, TAR: {tar/total*100:.0f}%. "
            f"Hypoglycemia episodes: {len(hypo_episodes)}. "
            f"Hyperglycemia episodes: {len(hyper_episodes)}."
        ),
        "glucose_value": float(np.mean(values)),
        "stats": {
            "mean": round(float(np.mean(values)), 1),
            "min": round(float(min(values)), 1),
            "max": round(float(max(values)), 1),
            "std": round(float(np.std(values)), 1),
            "tir_pct": round(tir/total*100, 1),
            "tbr_pct": round(tbr/total*100, 1),
            "tar_pct": round(tar/total*100, 1),
            "n_readings": total
        },
        "hypo_episodes": len(hypo_episodes),
        "hyper_episodes": len(hyper_episodes),
        "severity": "severe" if len(hypo_episodes) > 2 else "moderate" if len(hypo_episodes) > 0 else "normal"
    }

    return summary


# Demo: Consolidate today's working memory
today = datetime.date.today()
daily_summary = consolidate_daily_readings(
    working_mem.get_recent(),
    today,
    alice
)

print("📦 Memory Consolidation — Daily Summary:")
print(f"  Date: {daily_summary['date']}")
print(f"  Description: {daily_summary['description']}")
print(f"  Stats: {daily_summary['stats']}")
print(f"  Severity: {daily_summary['severity']}")

# Store consolidated summary in episodic memory
episodic_mem.store_episode(daily_summary)
print(f"\n✅ Consolidated daily summary stored in episodic memory ({len(episodic_mem)} total episodes)")

### Step 8: Temporal Decay Visualization

In [ ]:
# ─── Temporal Decay Analysis ──────────────────────────────────────────────────

def compute_temporal_decay(days_ago: float, lambda_: float = 0.1) -> float:
    """Compute temporal decay factor: exp(-lambda * days_ago)."""
    return math.exp(-lambda_ * days_ago)


# Show decay over time for different lambda values
print("⏳ Temporal Decay Analysis")
print("=" * 55)
print(f"{'Days Ago':>10} | {'λ=0.05':>8} | {'λ=0.10':>8} | {'λ=0.20':>8}")
print("-" * 55)

for days in [0, 1, 3, 7, 14, 30, 60, 90]:
    d05 = compute_temporal_decay(days, 0.05)
    d10 = compute_temporal_decay(days, 0.10)
    d20 = compute_temporal_decay(days, 0.20)
    bar = "█" * int(d10 * 20)
    print(f"{days:>10} | {d05:>8.3f} | {d10:>8.3f} | {d20:>8.3f}  {bar}")

print()
print("Interpretation:")
print("  λ=0.05: Slow decay — 30-day-old memory retains 22% weight")
print("  λ=0.10: Medium decay — 30-day-old memory retains 5% weight (GAS default)")
print("  λ=0.20: Fast decay — 30-day-old memory retains 0.2% weight")

# Show how temporal decay affects retrieval
print("\n📊 Effect of Temporal Decay on Episode Retrieval:")
print("=" * 65)

# Simulate two similar episodes: one recent, one old
similarity = 0.85  # Both have same semantic similarity

for days_ago in [0.5, 1, 3, 7, 30]:
    decay = compute_temporal_decay(days_ago, 0.1)
    final_score = similarity * decay
    bar = "█" * int(final_score * 30)
    print(f"  {days_ago:5.1f} days ago: sim={similarity:.2f} × decay={decay:.3f} = score={final_score:.3f}  {bar}")

### Step 9: Multi-Session Demo — 3 Days of Patient Data

In [ ]:
# ─── CGM Simulator ────────────────────────────────────────────────────────────

def simulate_cgm_day(
    date: datetime.date,
    base_glucose: float = 120.0,
    include_hypo: bool = False,
    include_hyper: bool = False,
    seed: int = 42
) -> List[GlucoseReading]:
    """
    Simulate a full day of CGM readings (288 readings at 5-min intervals).
    """
    random.seed(seed)
    np.random.seed(seed)

    readings = []
    glucose = base_glucose

    for i in range(288):  # 24h × 12 readings/hour
        hour = i / 12  # Hour of day (0-24)
        ts = datetime.datetime.combine(date, datetime.time(0, 0)) + datetime.timedelta(minutes=i*5)

        # Simulate meal effects
        if 7.5 <= hour <= 8.5:   # Breakfast
            glucose += random.uniform(5, 15)
        elif 12.5 <= hour <= 13.5:  # Lunch
            glucose += random.uniform(5, 20)
        elif 18.5 <= hour <= 19.5:  # Dinner
            glucose += random.uniform(10, 25)

        # Simulate insulin effect (gradual decrease)
        glucose -= random.uniform(0.5, 2.0)

        # Add noise
        glucose += random.gauss(0, 3)

        # Simulate hypoglycemia episode (3 AM)
        if include_hypo and 2.5 <= hour <= 4.0:
            glucose -= random.uniform(3, 8)

        # Simulate hyperglycemia (post-dinner)
        if include_hyper and 20.0 <= hour <= 22.0:
            glucose += random.uniform(5, 15)

        # Clamp to physiological range
        glucose = max(40, min(400, glucose))

        # Determine trend
        if len(readings) >= 3:
            recent_vals = [r.value for r in readings[-3:]]
            avg_change = (glucose - recent_vals[0]) / 3
            if avg_change > 2:
                trend = "rising"
            elif avg_change < -2:
                trend = "falling"
            else:
                trend = "stable"
        else:
            trend = "stable"

        readings.append(GlucoseReading(round(glucose, 1), ts, trend, "dexcom-g7-001"))

    return readings


print("✅ CGM simulator defined")

In [ ]:
# ─── 3-Day Memory Accumulation Demo ──────────────────────────────────────────

# Fresh memory stores for the demo
demo_working = WorkingMemory(capacity=24)
demo_episodic = EpisodicMemory(embedding_dim=64)
demo_semantic = SemanticMemory(alice)

today = datetime.date.today()

print("📅 3-Day Memory Accumulation Simulation")
print("=" * 60)

for day_offset in [2, 1, 0]:  # Day -2, Day -1, Today
    sim_date = today - datetime.timedelta(days=day_offset)
    day_label = ["Today", "Yesterday", "2 days ago"][day_offset]

    # Simulate different scenarios per day
    include_hypo = day_offset in [2, 0]   # Hypo on day -2 and today
    include_hyper = day_offset == 1        # Hyper on day -1

    day_readings = simulate_cgm_day(
        sim_date,
        base_glucose=115.0,
        include_hypo=include_hypo,
        include_hyper=include_hyper,
        seed=42 + day_offset
    )

    # Add last 2h of readings to working memory (for today)
    if day_offset == 0:
        for r in day_readings[-24:]:  # Last 2 hours
            demo_working.add(r)

    # Consolidate day into episodic memory
    summary = consolidate_daily_readings(day_readings, sim_date, alice)
    demo_episodic.store_episode(summary)

    # Update semantic memory
    if include_hypo:
        demo_semantic.set("patterns.typical_hypo_time", "nocturnal")

    # Print day summary
    stats = summary["stats"]
    print(f"\n{day_label} ({sim_date}):")
    print(f"  Readings: {stats['n_readings']} | Mean: {stats['mean']:.0f} mg/dL")
    print(f"  TIR: {stats['tir_pct']:.0f}% | TBR: {stats['tbr_pct']:.0f}% | TAR: {stats['tar_pct']:.0f}%")
    print(f"  Hypo episodes: {summary['hypo_episodes']} | Hyper episodes: {summary['hyper_episodes']}")
    print(f"  Severity: {summary['severity']}")

print(f"\n{'='*60}")
print(f"Memory state after 3 days:")
print(f"  Working memory: {len(demo_working)} readings (last 2h)")
print(f"  Episodic memory: {len(demo_episodic)} consolidated summaries")
print(f"  Semantic memory: {len(demo_semantic._graph)} top-level categories")

In [ ]:
# ─── Full GASMemory Integration ───────────────────────────────────────────────

class GASMemorySystem:
    """
    Unified memory system for the GAS Advisor Agent.
    Integrates all 3 active memory types.
    """

    def __init__(self, patient: PatientProfile, working_capacity: int = 24):
        self.working = WorkingMemory(capacity=working_capacity)
        self.episodic = EpisodicMemory(embedding_dim=64)
        self.semantic = SemanticMemory(patient)
        self.patient = patient

    def add_reading(self, reading: GlucoseReading):
        """Add a new CGM reading to working memory."""
        self.working.add(reading)

    def store_episode(self, episode: dict):
        """Store a clinical episode in episodic memory."""
        self.episodic.store_episode(episode)
        self.semantic.update_from_episode(episode)

    def consolidate_day(self, date: datetime.date):
        """Consolidate working memory into episodic memory."""
        readings = self.working.get_recent()
        if readings:
            summary = consolidate_daily_readings(readings, date, self.patient)
            self.episodic.store_episode(summary)
            return summary
        return None

    def build_context(self, current_state: dict, k_episodes: int = 3) -> str:
        """Build a rich context string for the LLM from all memory types."""
        parts = []

        # 1. Semantic memory (patient profile)
        parts.append("=== PATIENT PROFILE ===")
        parts.append(self.semantic.to_context_string())

        # 2. Working memory (recent readings)
        parts.append("\n=== RECENT GLUCOSE READINGS ===")
        parts.append(self.working.to_context_string())
        stats = self.working.get_stats()
        if stats:
            parts.append(f"Window stats: mean={stats['mean']}, min={stats['min']}, max={stats['max']}")

        # 3. Episodic memory (similar past episodes)
        similar = self.episodic.recall_similar_episodes(current_state, k=k_episodes)
        if similar:
            parts.append("\n=== SIMILAR PAST EPISODES ===")
            for ep in similar:
                parts.append(f"- [{ep['_days_ago']:.1f} days ago] {ep['description'][:100]}")
                if ep.get("outcome"):
                    parts.append(f"  Outcome: {ep['outcome'][:80]}")

        return "\n".join(parts)


# Demo: Build full context for the Advisor Agent
gas_memory = GASMemorySystem(alice)

# Add recent readings
for r in working_mem.get_recent():
    gas_memory.add_reading(r)

# Add past episodes
for ep in past_episodes:
    gas_memory.store_episode(ep)

# Build context for current critical state
current_state = {
    "type": "hypoglycemia",
    "description": "Critical hypoglycemia 45 mg/dL falling trend",
    "glucose_value": 45.0,
    "timestamp": datetime.datetime.now()
}

context = gas_memory.build_context(current_state, k_episodes=3)
print("🧠 GAS Advisor Agent — Full Memory Context:")
print("=" * 60)
print(context)

## 🏥 GAS Connection

The GAS system maintains **3 active memory stores** that work together:

```
CGM Reading (every 5 min)
        │
        ▼
Working Memory ──────────────────────────────────────────────┐
(last 2h readings)                                           │
        │ [every 24h: consolidate]                           │
        ▼                                                    │
Episodic Memory ─────────────────────────────────────────────┤
(past hypo/hyper episodes)                                   │
        │ [update patterns]                                  │
        ▼                                                    │
Semantic Memory ─────────────────────────────────────────────┤
(patient profile, preferences)                               │
                                                             │
                                                             ▼
                                                    GAS Advisor Agent
                                                    (RAG + Memory context)
```

### Memory in Action — Alice's Hypoglycemia

When Alice's glucose hits 45 mg/dL at 3 AM, the Advisor Agent's context includes:

1. **Working memory**: "Glucose has been falling for 2 hours: 140→120→95→70→55→45"
2. **Episodic memory**: "Alice had a similar nocturnal hypo 3 days ago at 3 AM. She was treated with glucose tablets and recovered in 20 minutes."
3. **Semantic memory**: "Alice has hypoglycemia unawareness — she cannot feel lows. Her preferred treatment is glucose tablets."

This context makes the recommendation **personalized and actionable** — not just generic ADA guidelines.

## ✅ Checkpoint

### What We Built

| Component | Type | Description |
|-----------|------|-------------|
| `WorkingMemory` | Working | Sliding window of last 24 CGM readings |
| `EpisodicMemory` | Episodic | Vector-indexed past clinical episodes |
| `SemanticMemory` | Semantic | Patient knowledge graph (profile, patterns) |
| `consolidate_daily_readings()` | Consolidation | Working → Episodic after 24h |
| `compute_temporal_decay()` | Decay | Score = similarity × exp(-λ × days) |
| `GASMemorySystem` | Integration | Unified memory for Advisor Agent |
| CGM Simulator | Utility | 3-day synthetic patient data |

### Key Concepts

- **4 memory types**: Working (fast), Episodic (vector), Semantic (graph), Procedural (weights)
- **Consolidation**: Prevents context overflow by summarizing working → episodic
- **Temporal decay**: Recent memories score higher than old ones
- **Shared memory**: All GAS agents read from the same GASMemorySystem

### What's Next

**Chapter 18: Agent Harness** — We'll build the production runtime that coordinates all 5 GAS agents:
- `GASHarness` class with tool dispatch and error handling
- Guardrails: block unsafe insulin dose recommendations
- Observability: log every tool call with timestamp and latency
- Context window management: sliding window + summarization

---
*Guardian Angel System | Part 5: Agentic AI | Chapter 17*